# Simulation 1 — Poisson loadings recovery in *gllvm*'s turf

Compare two estimators of a **fully dense** Poisson GLLVM loading matrix $W$
(and intercepts $b$) over a sweep of problem sizes, scoring each by the
orthogonal **Procrustes** error and by **bias / variance** of the recovered
parameters.

| | |
|---|---|
| latent dim | $q = 2$ |
| responses | $p \in \{10, 20, 50, 100\}$ |
| observations | $n \in \{20, 100, 500\}$ |
| | $\Rightarrow$ **12 settings**, each repeated $H$ times |

**Estimators** (fit to the *same* data each rep):

- **ZQE** — `ZQEAutoFitter`, Gaussian-MAP encoder, statistic $T=\log(1+y)$ on
  decoder *and* encoder — exactly the spec in [`../poisson.ipynb`](../poisson.ipynb).
- **gllvm** — R's `gllvm::gllvm()` (`method="VA"`) via `gllvm.r_gllvm.RGllvm`.

All the machinery lives in [`sweep.py`](sweep.py) (a driver specific to this
experiment, deliberately *not* in the `gllvm` source package). This notebook
just drives it. The analysis is in [`analysis.ipynb`](analysis.ipynb).

> Each rep generates its **own** true model and dataset from `seed == rep`, so
> every rep is independently reproducible. Results are written one CSV per
> (setting, rep) under `results/` — see
> [`results/DATA_DICTIONARY.md`](results/DATA_DICTIONARY.md).

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
# Make `import sweep` work regardless of the kernel's start directory.
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
if not os.path.exists(os.path.join(HERE, "sweep.py")):
    HERE = os.path.join(os.getcwd(), "simulations", "simulation_1")
sys.path.insert(0, HERE)

import torch
import sweep

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device      :", DEV)
print("q           :", sweep.Q)
print("p grid      :", sweep.P_GRID)
print("n grid      :", sweep.N_GRID)
print("ZQE config  :", sweep.ZQE_KW)
print("results dir :", sweep.RESULTS_DIR)

device      : cuda
q           : 2
p grid      : [10, 20, 50, 100]
n grid      : [20, 100, 500]
ZQE config  : {'steps_per_round': 150, 'max_rounds': 10, 'tol': 0.001, 'refine_lr': 0.5, 'warmup_lr': 0.5, 'ema_decay': 0.95, 'verbose': False}
results dir : /home/willwhite/GitHub/gllvm/simulations/simulation_1/results


## The wrapper

`sweep.run_setting(q, p, n, seed, device, rgllvm)` is the core unit: it
simulates one dense Poisson GLLVM + dataset from `seed`, fits both estimators,
Procrustes-rotates each loading estimate into the true gauge, and returns a
tidy long-format frame with the flattened parameters (loadings **and**
intercepts) for `true`, `zqe`, and `gllvm`, plus per-fit time / convergence /
Procrustes error / failure flag.

A quick demo on a small setting (not saved to `results/`):

In [2]:
from gllvm.r_gllvm import RGllvm
r = RGllvm(method="VA", family="poisson")
assert r.available(), f"R not found at {r.rscript!r} — set rscript=/workdir= for your install"

demo = sweep.run_setting(q=2, p=10, n=50, seed=0, device=DEV, rgllvm=r)
print("rows:", demo.shape, "| columns:", list(demo.columns))
demo.drop_duplicates("method")[["method", "failed", "time_sec",
                                "converged", "procrustes"]]

rows: (90, 14) | columns: ['q', 'p', 'n', 'rep', 'seed', 'method', 'failed', 'time_sec', 'converged', 'procrustes', 'param', 'i', 'j', 'value']


,method,failed,time_sec,converged,procrustes
0,true,NaN,NaN,NaN,NaN
30,zqe,0.0,8.098996,True,0.300798
60,gllvm,0.0,0.933067,NaN,0.287860


## Run the sweep

Resumable by construction: a rep whose CSV already exists is skipped, so this is
safe to re-run and **cheap to grow**. To go from $H=20$ to $H=100$ later, just
change `reps` below and re-run — only reps 20–99 are computed.

A failed fit (R error, NaN blow-up, …) is *flagged*, not fatal: the rep is still
written with that method's `failed=1.0` and `NaN` parameters, and the sweep
continues.

In [3]:
sweep.run_sweep(reps=20, device=DEV)

device=cuda  grid=12 settings × 20 reps = 240 fits;  240 done, 0 to run.


## Results overview

A quick sanity pass over what landed on disk. Detailed plots are in
[`analysis.ipynb`](analysis.ipynb).

In [4]:
import pandas as pd
df = sweep.load_results()

# one row per fit
fits = (df[df.method != "true"]
        .drop_duplicates(["p", "n", "rep", "method"])
        [["p", "n", "rep", "method", "failed", "time_sec", "procrustes"]])

print(f"reps per setting: {df.groupby(['p','n']).rep.nunique().min()}"
      f" – {df.groupby(['p','n']).rep.nunique().max()}")
print(f"failures: {int(fits.failed.sum())} / {len(fits)} fits")

summary = (fits.groupby(["p", "n", "method"])
           .agg(procrustes=("procrustes", "mean"),
                time_s=("time_sec", "mean"),
                fails=("failed", "sum"))
           .round(3))
summary

reps per setting: 20 – 20
failures: 1 / 480 fits


procrustes   time_s  fails
p   n   method                            
10  20  gllvm        0.445    0.700    0.0
        zqe          0.407    6.804    0.0
    100 gllvm        0.154    1.295    0.0
        zqe          0.169    6.477    0.0
    500 gllvm        0.180   28.455    0.0
        zqe          0.079    6.644    0.0
20  20  gllvm        0.468    0.784    0.0
        zqe          0.437    8.011    0.0
    100 gllvm        0.146    1.767    0.0
        zqe          0.166    6.413    0.0
    500 gllvm        0.083   52.849    0.0
        zqe          0.075    6.202    0.0
50  20  gllvm        0.405    1.300    1.0
        zqe          0.384    7.182    0.0
    100 gllvm        0.141    3.817    0.0
        zqe          0.180    6.587    0.0
    500 gllvm        0.062   90.525    0.0
        zqe          0.081    6.042    0.0
100 20  gllvm        0.498    2.376    0.0
        zqe          0.389    6.810    0.0
    100 gllvm        0.137   11.390    0.0
        zqe          0.173    6.747    0.0
    500 gllvm        0.881  154.883    0.0
        zqe          0.074    5.560    0.0